# 1 Transaction Data Generation

The purpose of this file is to take the transaction csv files provided on kaggle and generate 25 different monthly cohorts from Feb 2015 to Feb 2017. For each cohort, we stop at the last day of the preceeding month, then take all users that have a last expiration date in the cohort month. Then we aggregate their transaction information up to the cutoff point, keep track of their last transaction from the perspective of the cutoff point, and calculate whether or not they churned by seeing if the renew within 30 days of their last expiration date. Generating multiple cohorts to train and test on, we hope to create a model that is time agnostic. We pull inspiration for our churn label process from the scala script on the Kaggle competition website.

### 1. Set Up and File Loading


In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta
pd.set_option('display.max_columns', 100)

In [ ]:
DATA_DIR = '../data/'

In [ ]:
dtypes = {
    'msno': 'string',
    'payment_method_id': 'Int16',
    'payment_plan_days': 'Int16',
    'plan_list_price': 'Int16',
    'actual_amount_paid': 'Int16',
    'is_auto_renew': 'Int8',
    'is_cancel': 'Int8',
}

transactions_v1 = pd.read_csv(
    os.path.join(DATA_DIR, "raw/", "transactions.csv"),
    dtype = dtypes,
    parse_dates=[
        "transaction_date",
        "membership_expire_date"
    ]
)

transactions_v2 = pd.read_csv(
    os.path.join(DATA_DIR, "raw/", "transactions_v2.csv"),
    dtype = dtypes,
    parse_dates=[
        "transaction_date",
        "membership_expire_date"
    ]
)

#We combine the two data frames
transactions = pd.concat([transactions_v1, transactions_v2], ignore_index=True)
transactions.drop_duplicates(inplace = True)

print("Transactions shape:", transactions.shape)

Transactions shape: (22975416, 9)


## 2. Custom functions that will calculate churn, sort, and generate cohort data

In [ ]:
'''
This is the function we apply to calculate the gap between last expiration date
and next renewal. If greater than 30 days, the customer will be labeled as churned.
'''
def calculate_renewal_gap(group):
    last_expire_date = group.last_expire.iloc[0]

    gap = 9999

    for _, row in group.iterrows():
        trans_date = row["transaction_date"]
        expire_date = row["membership_expire_date"]

        #there are some transactions with expire date less than transaction date,
        #to avoid this messing up our gap calculation in this case we just set expire_date to the trans_date
        if(expire_date == datetime(1970, 1, 1)):
            expire_date= trans_date

        is_cancel = row["is_cancel"]

        #if the current row is an is_cancel row, their active expiration date should change to the new one
        if is_cancel == 1:
            if expire_date < last_expire_date:
                last_expire_date = expire_date
        #if we ever land on a non is_cancel row, we calculate the gap
        else:
            gap = (trans_date - last_expire_date).days
            break

    return gap


'''
This function sorts the transactions. Sorts by
#1. msno
#2. transaction date
#3. for transactions that occur on the same day, if they are for the same plan,
renew should come before cancel, if they are for different plans cancel should
come before renew.
#4. if both transactions are cancelations, earlier expiration date should come last
if both transactions are renewals, later expiration date should come last.

Note: for this function to sort correctly, it assume that there are only ever
2 transactions for a given user in a given day.
'''
def sort_transactions(df):
    temp_df = df.copy()

    # Convert dates to numeric for inversion
    temp_df["expire_numeric"] = temp_df["membership_expire_date"].astype("int64")

    #Count how many cancellations are in this day's pair (or single transaction) (0, 1, or 2)
    temp_df["cancel_sum"] = temp_df.groupby(["msno", "transaction_date"])[
        "is_cancel"
    ].transform("sum")

    # Are plan_list_price AND payment_plan_days identical for the pair (or single transaction), aka are
    # the transactions for the day for the same plan?
    temp_df["same_details"] = (
        temp_df.groupby(["msno", "transaction_date"])["plan_list_price"].transform(
            "nunique"
        )
        == 1
    ) & (
        temp_df.groupby(["msno", "transaction_date"])[
            "payment_plan_days"
        ].transform("nunique")
        == 1
    )

    # TIE BREAKER 1: For the "One True, One False" scenario
    # If details match: is_cancel = 0 first then is_cancel = 1 second
    # If details differ: is_cancel = 1 first then is_cancel = 0 second
    temp_df["tie_breaker_1"] = np.where(
        temp_df["same_details"], temp_df["is_cancel"], -temp_df["is_cancel"]
    )

    # TIE BREAKER 2: For the both is_cancel = 0 or both is_cancel = 1.
    # Both is_cancel = 1 (cancel_sum == 2): earlier expire comes last
    # Both False (cancel_sum == 0): later expire comes last
    temp_df["tie_breaker_2"] = np.where(
        temp_df["cancel_sum"] == 2,
        -temp_df["expire_numeric"],
        temp_df["expire_numeric"],
    )

    # Execute the sorting
    df_sorted = temp_df.sort_values(
        by=["msno", "transaction_date", "tie_breaker_1", "tie_breaker_2"],
        ascending=[True, True, True, True],
    )

    # Clean up all temporary helper columns
    df_sorted = df_sorted.drop(
        columns=[
            "expire_numeric",
            "cancel_sum",
            "same_details",
            "tie_breaker_1",
            "tie_breaker_2",
        ]
    )

    return df_sorted

'''
This function takes in sorted transaction data as well as the month and year of the
cut off date for the cohort. If we care about customers with expiration dates in February 2016
(from the perspective of the last transaction as of Jan 31), we would set the month to 2 and year
to 2016 as 2-01-2016 is the cutoff date.
For the specified cohort, this finds the users of interest, labels them as churned or not churned,
aggregates their historical transaction data up to the cutoff date, keeps track of their last
transaction, and returns the result.
'''

def cohort_transaction_data_generator(sorted_trans_data: pd.DataFrame, month, year):
    #This is the cohort cut off date. We break the data into history data and future data
    #based on it. We only need two months of future data to properly calculate churn.
    #The entire month "month" and then the entire month "month + 1" We use a timedelta of 65 days
    #to add an extra buffer of 5 days into the future data.
    history_cutoff = datetime(year, month, 1)
    history_data = sorted_trans_data[sorted_trans_data['transaction_date'] < history_cutoff]
    future_data = sorted_trans_data[(sorted_trans_data["transaction_date"] >= history_cutoff) &
                                    (sorted_trans_data['transaction_date'] <= history_cutoff + timedelta(65))]

    #We find the last transaction for each user
    last_transaction = history_data.groupby('msno', as_index = False).last()

    #we find the next_month cohort cutoff date
    next_month = history_cutoff
    if(month == 12):
        next_month = datetime(year + 1, 1, 1)
    else:
        next_month = datetime(year, month + 1, 1)

    #our prediction candidates are those users from the perspective of history_cutoff
    #have a last expiration date in the cohort month in question
    prediction_candidates = last_transaction.loc[
    (last_transaction["membership_expire_date"] >= history_cutoff)
    & (last_transaction["membership_expire_date"] < next_month)
    ][['msno', 'membership_expire_date']]
    prediction_candidates.columns = ['msno', 'last_expire']

    # Left outer join prediction_candidates with future data
    joined_data = pd.merge(
    prediction_candidates, future_data, on="msno", how="left"
    )

    #any prediction_candidate with no future data is an obvious churner
    no_activity = joined_data[joined_data["payment_method_id"].isna()].copy()
    no_activity["is_churn"] = 1

    #any prediction_candidate with activity we calculate gap to renewal
    has_activity = joined_data[joined_data["payment_method_id"].notna()].copy()
    renewal_gap = (
    has_activity.groupby(["msno"])
    .apply(calculate_renewal_gap, include_groups=False)
    .reset_index(name = 'gap')
    )

    #if they renew within 30 days they did not churn
    valid_renewals = renewal_gap[renewal_gap["gap"] <= 30].copy()
    valid_renewals["is_churn"] = 0

    #if they renew after 30 days they did churn
    late_renewals = renewal_gap[renewal_gap["gap"] > 30].copy()
    late_renewals["is_churn"] = 1

    #we concat together the three dataframes that contain churn labels for
    #different customer behaviors
    churn_label =  pd.concat(
    [
        valid_renewals[["msno", "is_churn"]],
        late_renewals[["msno", "is_churn"]],
        no_activity[["msno", "is_churn"]],
    ],
    ignore_index=True,
    )

    #Next, we modify the last_transaction dataframe dropping the payment_method_id feature
    #and renaming the columns.
    modified_last_transaction = last_transaction.copy()
    modified_last_transaction.drop(columns = ['payment_method_id'], inplace = True)
    modified_last_transaction.columns = ['msno', 'last_payment_plan_days',
                                         'last_plan_list_price',
                                         'last_actual_amount_paid',
                                         'last_is_auto_renew',
                                         'last_transaction_date',
                                         'last_expire', 'last_is_cancel']

    #we aggregate the transaction data for each user in history data
    trans_agg = history_data.groupby('msno').agg(
    num_transactions=('transaction_date', 'count'),
    total_paid=('actual_amount_paid', 'sum'),
    avg_plan_price=('plan_list_price', 'mean'),
    total_auto_renew=('is_auto_renew', 'sum'),
    total_cancel=('is_cancel', 'sum'),
    first_transaction_date=('transaction_date', 'min')
    ).reset_index()

    #merging left with churn_label, adds the aggregated transaction data for only
    #the customers we care about
    trans_agg = pd.merge(churn_label, trans_agg, on = 'msno', how = 'left')
    #then merge to add in last transaction data for the customers we care about
    trans_agg = pd.merge(trans_agg, modified_last_transaction, on = 'msno', how = 'left')

    #Now compute some features of interest
    trans_agg['days_since_first_trans'] = (history_cutoff - trans_agg['first_transaction_date']).dt.days
    trans_agg['days_since_last_trans'] = (history_cutoff - trans_agg['last_transaction_date']).dt.days
    trans_agg['days_to_expire'] = (trans_agg['last_expire']- history_cutoff).dt.days
    trans_agg['avg_payment_per_day'] = trans_agg['total_paid'] / ((trans_agg['days_since_first_trans'] + trans_agg['days_to_expire']))

    #We drop first_transaction_date as it is a column we are no longer interested in
    trans_agg.drop(columns=['first_transaction_date'], inplace=True)

    return trans_agg



## 3. Prep The Data

Since the data from kaggle is messy and there are some users with multiple transactions in a single day where it is impossible to properly sort them, we are making the executive decision to drop these users. Moving forward, we would design a better transaction labeling procedure so that the order of transactions is easier to determine. Like maybe when logging the data for   transaction there is a user transaction count.

This dropping of users is unfortunate but it is a small number compared to the
total amount of users we have.

We also sort the resulting transaction dataframe.

In [ ]:
data_day_agg = transactions.groupby(['msno', 'transaction_date'], as_index = False).agg(num_transactions = ('payment_method_id', 'count'))
mult_users = data_day_agg.loc[data_day_agg.num_transactions > 2].msno.unique()
transactions = transactions[~transactions["msno"].isin(mult_users)]
sorted_transactions = sort_transactions(transactions)

## 4. Generate the Cohort Transaction Data and the Churn Labels


In [ ]:
#Now let's generate cohort data

#Define a worker function for a single (month, year) combination
def process_cohort(month, year):
    print(f"Processing: {month}/{year}")

    # Generate the data
    cohort_data = cohort_transaction_data_generator(sorted_transactions, month, year)
    # We add in a cohort_cutoff_date feature to keep track of the cohort
    cohort_data['cohort_cutoff_date'] = datetime(year, month, 1)

    return cohort_data

#Run a loop to get all the desired cohorts
results = []
for year in range(2015, 2018):
    for month in range(1, 13):
        # Skip condition, we don't calculate for the first month we have since
        #this is too soon to the beginning of our data
        if year == 2015 and month == 1:
            continue

        # Break condition, we break here as this is the end of our scope
        if year == 2017 and month == 3:
            break

        results.append(process_cohort(month, year))

#Combine everything into one final DataFrame
train = pd.concat(results, ignore_index=True)

#Save the result as a csv
train.to_csv(
    os.path.join(DATA_DIR, "processed/", "mult_cohort_transaction_data.csv"), index=False
)


Processing: 2/2015
Processing: 3/2015
Processing: 4/2015
Processing: 5/2015
Processing: 6/2015
Processing: 7/2015
Processing: 8/2015
Processing: 9/2015
Processing: 10/2015
Processing: 11/2015
Processing: 12/2015
Processing: 1/2016
Processing: 2/2016
Processing: 3/2016
Processing: 4/2016
Processing: 5/2016
Processing: 6/2016
Processing: 7/2016
Processing: 8/2016
Processing: 9/2016
Processing: 10/2016
Processing: 11/2016
Processing: 12/2016
Processing: 1/2017
Processing: 2/2017
